# 07: Geocoding & Address Processing

This notebook demonstrates geocoding capabilities in siege_utilities:

1. **Address to Coordinates** - Convert addresses to lat/lon
2. **Address Concatenation** - Build standardized addresses
3. **Batch Geocoding** - Process multiple addresses
4. **Country Code Filtering** - Restrict results by country

## Prerequisites

```bash
pip install -e /path/to/siege_utilities
pip install geopy
```

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import siege_utilities as su
su.configure_shared_logging(level="INFO")

from siege_utilities.geo.geocoding import (
    get_coordinates,
    concatenate_addresses,
    use_nominatim_geocoder
)

import pandas as pd
import time

su.log_info("Geocoding imports successful!")

## 1. Single Address Geocoding

Convert an address string to latitude/longitude coordinates.

In [ ]:
# Geocode a single address
address = "1600 Pennsylvania Avenue NW, Washington, DC"

su.log_info(f"Geocoding: {address}")
coords = get_coordinates(address, country_codes='us')

if coords:
    lat, lon = coords  # Returns tuple (lat, lon)
    su.log_info(f"Result: latitude={lat:.6f}, longitude={lon:.6f}")
else:
    su.log_warning("No results found")

In [ ]:
# More examples
test_addresses = [
    "Empire State Building, New York, NY",
    "Golden Gate Bridge, San Francisco, CA",
    "Space Needle, Seattle, WA"
]

su.log_info("Geocoding landmarks:")
su.log_info("-" * 60)
for addr in test_addresses:
    # Rate limit - Nominatim requires 1 second between requests
    time.sleep(1)
    coords = get_coordinates(addr, country_codes='us')
    if coords:
        lat, lon = coords  # Returns tuple (lat, lon)
        su.log_info(f"{addr[:40]:40} -> ({lat:.4f}, {lon:.4f})")
    else:
        su.log_warning(f"{addr[:40]:40} -> Not found")

## 2. Address Concatenation

Build standardized address strings from components.

In [ ]:
# Build address from components
address = concatenate_addresses(
    street="123 Main Street",
    city="San Francisco",
    state_province_area="California",
    postal_code="94102",
    country="USA"
)

su.log_info(f"Concatenated address: {address}")

In [ ]:
# Partial address (some fields missing)
partial_address = concatenate_addresses(
    city="Los Angeles",
    state_province_area="CA"
)

su.log_info(f"Partial address: {partial_address}")

## 3. Batch Geocoding

Geocode multiple addresses with rate limiting.

In [ ]:
# Sample addresses to geocode
addresses_df = pd.DataFrame({
    'name': ['City Hall', 'Airport', 'University'],
    'street': ['200 N Spring St', '1 World Way', '405 Hilgard Ave'],
    'city': ['Los Angeles', 'Los Angeles', 'Los Angeles'],
    'state': ['CA', 'CA', 'CA']
})

su.log_info("Sample addresses:")
su.log_info(f"\n{addresses_df}")

In [ ]:
def batch_geocode(df, street_col, city_col, state_col, delay=1.0):
    """
    Batch geocode addresses in a DataFrame.
    
    Args:
        df: DataFrame with address columns
        street_col: Column name for street address
        city_col: Column name for city
        state_col: Column name for state
        delay: Seconds between requests (Nominatim requires >= 1)
        
    Returns:
        DataFrame with lat/lon columns added
    """
    result_df = df.copy()
    result_df['latitude'] = None
    result_df['longitude'] = None
    result_df['geocode_status'] = None
    
    for idx, row in result_df.iterrows():
        # Build address
        address = concatenate_addresses(
            street=row[street_col],
            city=row[city_col],
            state_province_area=row[state_col]
        )
        
        # Geocode - returns tuple (lat, lon) or None
        coords = get_coordinates(address, country_codes='us')
        
        if coords:
            lat, lon = coords
            result_df.at[idx, 'latitude'] = lat
            result_df.at[idx, 'longitude'] = lon
            result_df.at[idx, 'geocode_status'] = 'success'
        else:
            result_df.at[idx, 'geocode_status'] = 'not_found'
        
        # Rate limit
        time.sleep(delay)
    
    return result_df

# Geocode the addresses
su.log_info("Batch geocoding (this may take a few seconds)...")
geocoded_df = batch_geocode(addresses_df, 'street', 'city', 'state')

su.log_info("Results:")
su.log_info(f"\n{geocoded_df[['name', 'latitude', 'longitude', 'geocode_status']]}")

## 4. Using Nominatim Directly

The `use_nominatim_geocoder` function provides more control.

In [ ]:
# Get detailed geocoding result
# use_nominatim_geocoder returns a JSON string with detailed info
import json

result_str = use_nominatim_geocoder(
    query_address="Statue of Liberty, New York",
    country_codes='us'
)

if result_str:
    result = json.loads(result_str)
    su.log_info("Geocoding result:")
    for key, value in result.items():
        su.log_info(f"  {key}: {value}")
else:
    su.log_warning("No results found")

## 5. Visualize Geocoded Points

Plot geocoded points on a map.

In [ ]:
import matplotlib.pyplot as plt
from siege_utilities.geo.spatial_data import get_census_boundaries

# Get California boundary for context
states = get_census_boundaries(year=2020, geographic_level='state')
ca = states[states['statefp'] == '06']

# Filter to successful geocodes
geocoded_points = geocoded_df[geocoded_df['geocode_status'] == 'success']

if len(geocoded_points) > 0:
    fig, ax = plt.subplots(figsize=(8, 10))
    
    # Plot state boundary
    ca.plot(ax=ax, facecolor='lightgray', edgecolor='black')
    
    # Plot points
    ax.scatter(
        geocoded_points['longitude'],
        geocoded_points['latitude'],
        c='red', s=100, zorder=5, marker='o'
    )
    
    # Add labels
    for idx, row in geocoded_points.iterrows():
        ax.annotate(
            row['name'],
            xy=(row['longitude'], row['latitude']),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=8
        )
    
    # Focus on LA area
    ax.set_xlim(-119, -117)
    ax.set_ylim(33, 35)
    ax.set_title('Geocoded Locations in Los Angeles')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    
    plt.tight_layout()
    plt.savefig('/tmp/geocoded_points.png', dpi=100, bbox_inches='tight')
    su.log_info("Map saved to /tmp/geocoded_points.png")
    plt.show()
else:
    su.log_warning("No geocoded points to visualize")

## Summary

| Function | Purpose |
|----------|----------|
| `get_coordinates()` | Simple address to lat/lon |
| `concatenate_addresses()` | Build address strings |
| `use_nominatim_geocoder()` | Detailed geocoding with options |

**Important Notes:**
- Nominatim (OpenStreetMap) requires 1 second delay between requests
- Use `country_codes` parameter to improve accuracy
- For high-volume geocoding, consider commercial services